**Upload and Preprocess Dataste**

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import warnings

warnings.filterwarnings("ignore")

# Define URLs for each client's dataset
client_1_url = "https://raw.githubusercontent.com/Me-Rajdip/Data-Set/refs/heads/main/VFL%20Dataset/Modified_Cardiovascular_Disease_Dataset.csv"
client_2_url = "https://raw.githubusercontent.com/Me-Rajdip/Data-Set/refs/heads/main/VFL%20Dataset/heart.csv"
client_3_url = "https://raw.githubusercontent.com/Me-Rajdip/Data-Set/refs/heads/main/VFL%20Dataset/Cardiovascular_Disease_Dataset.csv"

# Load the datasets
client_1_df = pd.read_csv(client_1_url)
client_2_df = pd.read_csv(client_2_url)
client_3_df = pd.read_csv(client_3_url)

# Drop "Unnamed" column from client_1_df
client_1_df = client_1_df.loc[:, ~client_1_df.columns.str.contains('^Unnamed')]

# Drop "patientid" column from client_3_df
client_3_df = client_3_df.drop(columns=["patientid"], errors='ignore')

# Target variable for each client
client_1_target = 'target'
client_2_target = 'HeartDisease'
client_3_target = 'target'

# Separate numeric and categorical columns for Client 2
client_2_numeric_cols = client_2_df.select_dtypes(include=[np.number]).columns.tolist()
client_2_categorical_cols = client_2_df.select_dtypes(exclude=[np.number]).columns.tolist()

# Fill missing values for numeric columns with median
client_1_df = client_1_df.fillna(client_1_df.median())
client_2_df[client_2_numeric_cols] = client_2_df[client_2_numeric_cols].fillna(client_2_df[client_2_numeric_cols].median())
client_3_df = client_3_df.fillna(client_3_df.median())

# Apply one-hot encoding for categorical columns in Client 2
encoder = OneHotEncoder(drop='first', sparse_output=False)
client_2_encoded = pd.DataFrame(encoder.fit_transform(client_2_df[client_2_categorical_cols]))
client_2_encoded.columns = encoder.get_feature_names_out(client_2_categorical_cols)
# Drop original categorical columns and concatenate the encoded ones
client_2_df = pd.concat([client_2_df[client_2_numeric_cols], client_2_encoded], axis=1)

# Split the data into train and test sets
client_1_train, client_1_test = train_test_split(client_1_df, test_size=0.2, random_state=42)
client_2_train, client_2_test = train_test_split(client_2_df, test_size=0.2, random_state=42)
client_3_train, client_3_test = train_test_split(client_3_df, test_size=0.2, random_state=42)

# Prepare datasets for training (no alignment needed, just use the existing features)
def prepare_dataset(df, features, target):
    scaler = StandardScaler()
    X = scaler.fit_transform(df[features])
    y = df[target].values
    return {"features": X, "labels": y}

client_1_data = prepare_dataset(client_1_train, client_1_df.columns.difference([client_1_target]), client_1_target)
client_2_data = prepare_dataset(client_2_train, client_2_df.columns.difference([client_2_target]), client_2_target)
client_3_data = prepare_dataset(client_3_train, client_3_df.columns.difference([client_3_target]), client_3_target)


**VFL + SMPC**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from docx import Document
import matplotlib.pyplot as plt
import seaborn as sns
import os

def generate_shares(value, prime=2**31-1):
    """Generate 3 additive shares for each value element."""
    value = np.array(value).flatten()
    shares = []
    for v in value:
        s0 = np.random.randint(0, prime) % prime
        s1 = np.random.randint(0, prime) % prime
        s2 = (v - (s0 + s1)) % prime  # Ensure modular arithmetic
        shares.append([s0, s1, s2])
    return np.array(shares).T  # Transpose to get 3 shares per client

def reconstruct_secret(shares, prime=2**31-1):
    """Reconstruct the original secret from additive shares."""
    if len(shares) < 3:
        raise ValueError("At least 3 shares are required for reconstruction")
    
    # Sum all shares element-wise
    reconstructed = np.sum(shares[:3], axis=0) % prime
    
    # Handle potential overflow by ensuring values are within the proper range
    half_prime = prime // 2
    reconstructed = np.where(reconstructed > half_prime, reconstructed - prime, reconstructed)
    
    return reconstructed

def logistic_regression_local_update(dataset, weights, bias, learning_rate=0.1, num_iterations=100):
    model = LogisticRegression(max_iter=num_iterations, solver='lbfgs')
    model.fit(dataset["features"], dataset["labels"])
    weights = model.coef_.flatten()
    bias = model.intercept_[0]
    return weights, bias

def clip_update(update, threshold):
    norm = np.linalg.norm(update)
    if norm > threshold:
        return update * (threshold / norm)
    else:
        return update

def evaluate_model(dataset, weights, bias):
    X = dataset["features"]
    y = dataset["labels"]
    linear_model = np.dot(X, weights) + bias
    y_pred = (1 / (1 + np.exp(-linear_model))) >= 0.5
    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, zero_division=0)
    rec = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)
    return acc, prec, rec, f1

def save_results_and_plots(results_df, num_clients, num_rounds):
    # Create directory
    if not os.path.exists("VFL_With_SMPC"):
        os.makedirs("VFL_With_SMPC")

    # Save results as Word file
    doc = Document()
    doc.add_heading('Federated Learning Results (With SMPC)', 0)
    
    # Add summary statistics
    doc.add_heading('Summary Statistics', level=1)
    final_results = results_df[results_df['Round'] == results_df['Round'].max()]
    doc.add_paragraph(f"Final Global Model Accuracy: {final_results[final_results['Client'] == 'Global Model']['Accuracy'].values[0]:.4f}")
    doc.add_paragraph(f"Final Global Model Precision: {final_results[final_results['Client'] == 'Global Model']['Precision'].values[0]:.4f}")
    doc.add_paragraph(f"Final Global Model Recall: {final_results[final_results['Client'] == 'Global Model']['Recall'].values[0]:.4f}")
    doc.add_paragraph(f"Final Global Model F1 Score: {final_results[final_results['Client'] == 'Global Model']['F1 Score'].values[0]:.4f}")
    
    # Add detailed results table
    doc.add_heading('Detailed Results', level=1)
    table = doc.add_table(rows=1, cols=len(results_df.columns))
    table.style = 'Table Grid'
    hdr_cells = table.rows[0].cells
    for i, col in enumerate(results_df.columns):
        hdr_cells[i].text = col
    for _, row in results_df.iterrows():
        row_cells = table.add_row().cells
        for i, value in enumerate(row):
            row_cells[i].text = str(value)
    doc.save("VFL_With_SMPC/results.docx")

    # Prepare data for plotting
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
    client_data = {f'Client {i+1}': {'Accuracy': [], 'Precision': [], 'Recall': [], 'F1 Score': []} 
                  for i in range(num_clients)}
    global_data = {'Accuracy': [], 'Precision': [], 'Recall': [], 'F1 Score': []}
    
    for round_num in range(1, num_rounds + 1):
        round_data = results_df[results_df['Round'] == round_num]
        
        # Get global model metrics
        global_round = round_data[round_data['Client'] == 'Global Model']
        if not global_round.empty:
            for metric in metrics:
                global_data[metric].append(global_round[metric].values[0])
        
        # Get client metrics
        for i in range(num_clients):
            client_round = round_data[round_data['Client'] == i+1]
            if not client_round.empty:
                for metric in metrics:
                    client_data[f'Client {i+1}'][metric].append(client_round[metric].values[0])

    # Plot metrics
    rounds = list(range(1, num_rounds + 1))
    colors = ['red', 'green', 'blue', 'orange', 'purple'][:num_clients]

    def plot_metric(metric, title, ylabel, filename):
        plt.figure(figsize=(10, 6))
        for i, client in enumerate(client_data.keys()):
            plt.plot(rounds, client_data[client][metric][:num_rounds], 
                    marker='o', color=colors[i], label=client)
        plt.plot(rounds, global_data[metric][:num_rounds], 
                marker='s', color='black', linestyle='--', label='Global Model')
        plt.xlabel("Round Number")
        plt.ylabel(ylabel)
        plt.title(title)
        plt.legend()
        plt.grid(True)
        plt.savefig(f"VFL_With_SMPC/{filename}.png")
        plt.close()

    plot_metric('Accuracy', "Accuracy Over Rounds", "Accuracy", "accuracy")
    plot_metric('Precision', "Precision Over Rounds", "Precision", "precision")
    plot_metric('Recall', "Recall Over Rounds", "Recall", "recall")
    plot_metric('F1 Score', "F1 Score Over Rounds", "F1 Score", "f1_score")

def federated_learning_Using_LR_SMPC(
    datasets, initial_weights, initial_biases, learning_rate, clipping_threshold, num_rounds
):
    weights = [w.copy() for w in initial_weights]
    biases = [b for b in initial_biases]
    results = []
    best_global_acc = 0.0
    patience = 3
    no_improvement = 0
    n_clients = len(datasets)  
    prime = 2**31-1  # Mersenne prime for modular arithmetic

    for round_num in range(1, num_rounds + 1):
        local_weight_updates = []
        local_bias_updates = []
        local_clipped_updates = []
        target_accuracies = []

        print(f"\n=== Round {round_num} ===")

        # Client updates
        for i, dataset in enumerate(datasets):
            print(f"\n--- Client {i+1} ---")
            if round_num == 1:
                # Initial training on dataset
                print("Initial training on local dataset...")
                weight_update, bias_update = logistic_regression_local_update(
                    dataset, weights[i], biases[i], learning_rate
                )
            else:
                # Subsequent rounds: Use previous adjusted weights/biases directly
                print("Using adjusted weights and biases from the previous round...")
                weight_update = weights[i].copy()
                bias_update = biases[i].copy()

            clipped_weight = clip_update(weight_update, clipping_threshold)
            clipped_bias = clip_update(np.array([bias_update]), clipping_threshold)[0]

            # Evaluate before sharing and store target accuracy
            acc, prec, rec, f1 = evaluate_model(dataset, clipped_weight, clipped_bias)
            target_accuracies.append(acc)
            local_clipped_updates.append((clipped_weight, clipped_bias))

            # Generate additive shares for the clipped updates
            print("Generating additive secret shares for clipped updates...")
            weight_shares = generate_shares(clipped_weight, prime)
            bias_shares = generate_shares(np.array([clipped_bias]), prime)  # Bias as array for consistency

            print(f"\nClient {i+1} Weight Shares:")
            for j in range(3):
                print(f"Share {j}: {weight_shares[j]}")
            print(f"Client {i+1} Bias Shares:")
            for j in range(3):
                print(f"Share {j}: {bias_shares[j][0]}")

            # Store shares for aggregation
            local_weight_updates.append(weight_shares)
            local_bias_updates.append(bias_shares)

            # Log before sharing
            results.append({
                "Round": round_num, "Client": i+1, "Stage": "Before Sharing",
                "Weights": weight_update.tolist(), "Bias": bias_update,
                "Accuracy": acc, "Precision": prec, "Recall": rec, "F1 Score": f1
            })

        # Secure aggregation using SMPC
        print("\n--- Secure Aggregation (SMPC) ---")
        
        # New share distribution: Each client j receives the j-th share from every client's update.
        received_weight_shares = []  # will be a list of lists: for each client j, a list of received weight shares
        received_bias_shares = []    # for each client j, a list of received bias shares
        
        for j in range(n_clients):
            client_weight_shares = []
            client_bias_shares = []
            for i in range(n_clients):
                # From client i, send share with index j to client j.
                client_weight_shares.append(local_weight_updates[i][j])
                # For bias, local_bias_updates[i] shape is (3,1); extract scalar.
                client_bias_shares.append(local_bias_updates[i][j][0])
            received_weight_shares.append(client_weight_shares)
            received_bias_shares.append(client_bias_shares)
        
        # Print received shares for verification
        for j in range(n_clients):
            print(f"\nClient {j+1} received weight shares:")
            for src in range(n_clients):
                print(f"From Client {src+1}: {received_weight_shares[j][src]}")
            print(f"Client {j+1} received bias shares:")
            for src in range(n_clients):
                print(f"From Client {src+1}: {received_bias_shares[j][src]}")

        # Step 2: Compute local sums at each client
        print("\nStep 2: Computing local sums at each client...")
        sum_weights_shares = []
        sum_biases_shares = []
        
        for j in range(n_clients):
            weight_sum = np.zeros_like(local_weight_updates[0][0])
            for share in received_weight_shares[j]:
                weight_sum = (weight_sum + share)
            sum_weights_shares.append(weight_sum)
            
            bias_sum = 0
            for share in received_bias_shares[j]:
                bias_sum = (bias_sum + share)
            sum_biases_shares.append(bias_sum)
            
            print(f"\nClient {j+1} local sum of weight shares: {weight_sum}")
            print(f"Client {j+1} local sum of bias shares: {bias_sum}")
        
        # Step 3: Reconstruct the aggregated sums
        print("\nStep 3: Reconstructing the aggregated sums...")
        aggregated_weight = reconstruct_secret(sum_weights_shares, prime)
        aggregated_bias = reconstruct_secret(sum_biases_shares, prime)
        
        print(f"\nReconstructed Aggregated Weight: {aggregated_weight}")
        print(f"Reconstructed Aggregated Bias: {aggregated_bias}")

        # Log aggregated model
        combined_data = {
            "features": np.vstack([d["features"] for d in datasets]),
            "labels": np.hstack([d["labels"] for d in datasets])
        }
        acc_agg, prec_agg, rec_agg, f1_agg = evaluate_model(combined_data, aggregated_weight, aggregated_bias)
        results.append({
            "Round": round_num, "Client": "Global Server", "Stage": "Aggregated (SMPC)",
            "Weights": aggregated_weight.tolist(), "Bias": aggregated_bias,
            "Accuracy": acc_agg, "Precision": prec_agg,
            "Recall": rec_agg, "F1 Score": f1_agg
        })

        # Client adjustment phase
        print("\n--- Client Adjustment Phase ---")
        for i in range(len(datasets)):
            dataset = datasets[i]
            print(f"\nAdjusting Client {i+1}...")

            # Get THIS ROUND'S before-sharing accuracy as the target
            target_acc = target_accuracies[i]

            # Get original clipped update (before sharing) for this client
            clipped_weight, clipped_bias = local_clipped_updates[i]

            # Compute delta: Difference between client's clipped model and aggregated global
            delta_weight = clipped_weight - aggregated_weight
            delta_bias = clipped_bias - aggregated_bias

            # Line search for alpha to reach target accuracy with clipped models
            print("Performing line search for alpha...")
            best_alpha = 0.0
            best_acc = 0.0
            for alpha in np.linspace(0, 1, 11):
                w_test = aggregated_weight + alpha * delta_weight
                b_test = aggregated_bias + alpha * delta_bias
                acc, _, _, _ = evaluate_model(dataset, w_test, b_test)
                print(f"Alpha {alpha:.1f}: Accuracy {acc:.4f}")

                if acc >= target_acc:
                    best_alpha = alpha
                    best_acc = acc
                    break
                if acc > best_acc:
                    best_alpha = alpha
                    best_acc = acc

            print(f"Best Alpha: {best_alpha}, Best Accuracy: {best_acc}")

            # Compute new weights and bias without perturbation
            new_weight = aggregated_weight + best_alpha * delta_weight
            new_bias = aggregated_bias + best_alpha * delta_bias

            # Log adjusted model
            acc_adj, prec_adj, rec_adj, f1_adj = evaluate_model(dataset, new_weight, new_bias)
            results.append({
                "Round": round_num, "Client": i+1, "Stage": "Adjusted",
                "Weights": new_weight.tolist(), "Bias": new_bias,
                "Accuracy": acc_adj, "Precision": prec_adj, "Recall": rec_adj, "F1 Score": f1_adj
            })

            # Update client weights and biases with adjusted versions
            weights[i] = new_weight.copy()
            biases[i] = new_bias

        # Evaluate global model (average of client models)
        global_weights = np.mean(weights, axis=0)
        global_bias = np.mean(biases)
        acc_global, prec_global, rec_global, f1_global = evaluate_model(combined_data, global_weights, global_bias)
        results.append({
            "Round": round_num, "Client": "Global Model", "Stage": "Post Adjustment",
            "Weights": global_weights.tolist(), "Bias": global_bias,
            "Accuracy": acc_global, "Precision": prec_global, "Recall": rec_global, "F1 Score": f1_global
        })

        # Early stopping
        if acc_global > best_global_acc + 0.0001:
            best_global_acc = acc_global
            no_improvement = 0
        else:
            no_improvement += 1
        if no_improvement >= patience:
            print(f"Early stopping at round {round_num}")
            break

    # Final evaluation
    final_weights = np.mean(weights, axis=0)
    final_bias = np.mean(biases)
    acc_final, prec_final, rec_final, f1_final = evaluate_model(combined_data, final_weights, final_bias)
    print(f"\nFinal Global Model Accuracy: {acc_final:.4f}")
    print(f"Final Global Model Precision: {prec_final:.4f}")
    print(f"Final Global Model Recall: {rec_final:.4f}")
    print(f"Final Global Model F1 Score: {f1_final:.4f}")
    
    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    
    # Save results and plots
    save_results_and_plots(results_df, n_clients, min(num_rounds, len(results_df['Round'].unique())))
    
    return results_df

def pad_features_to_max_size(datasets):
    max_features = max(d["features"].shape[1] for d in datasets)
    for d in datasets:
        pad_width = max_features - d["features"].shape[1]
        if pad_width > 0:
            d["features"] = np.hstack([d["features"], np.zeros((d["features"].shape[0], pad_width))])
    return datasets, max_features


datasets = [client_1_data, client_2_data, client_3_data]
    
# Pad datasets and initialize weights
datasets, max_features = pad_features_to_max_size(datasets)
initial_weights = [np.zeros(max_features) for _ in datasets]
initial_biases = [0.0 for _ in datasets]

# Run Federated Learning with SMPC
results_df = federated_learning_Using_LR_SMPC(
    datasets=datasets,
    initial_weights=initial_weights,
    initial_biases=initial_biases,
    learning_rate=0.1,
    clipping_threshold=3.0,
    num_rounds=10
)


=== Round 1 ===

--- Client 1 ---
Initial training on local dataset...
Generating additive secret shares for clipped updates...

Client 1 Weight Shares:
Share 0: [6.28850904e+08 1.02733733e+09 2.07362389e+09 6.34433055e+08
 2.06940682e+09 2.02132852e+09 1.95145023e+09 1.79844288e+09
 1.97417542e+09 1.84836956e+09 5.53699492e+08 1.01284134e+09
 6.43995774e+08 3.14503611e+08 1.54347386e+09 1.63395542e+09
 3.74510371e+08]
Share 1: [1.17543892e+09 1.78357395e+09 3.39915890e+07 3.14161821e+08
 2.12077418e+08 1.58140871e+09 6.25056315e+08 3.29543481e+08
 6.18546960e+08 2.36653693e+08 9.78326682e+08 7.97616782e+08
 9.94839822e+08 1.69187197e+09 1.80971954e+09 7.14492493e+08
 1.59683890e+09]
Share 2: [3.43193822e+08 1.48405601e+09 3.98681688e+07 1.19888877e+09
 2.01348306e+09 6.92230069e+08 1.71846075e+09 1.94972916e+07
 1.70224492e+09 6.24603990e+07 6.15457473e+08 3.37025531e+08
 5.08648051e+08 1.41108067e+08 9.41773898e+08 1.94651939e+09
 1.76134374e+08]
Client 1 Bias Shares:
Share 0: 16076